# Cvičení 2: Rekurze a rekurzivní datové typy, funkce vyššího řádu

## HOF
„Funkce vyššího řádu“ (higher-order functions) je odborně znějící termín pro funkci, která jako parametr bere jinou funkci (a tu potom typicky ve svém těle nějak používá):

In [1]:
flipArgs :: (a -> b -> c) -> b -> a -> c
flipArgs f x y = f y x
--                      f -> x -> y -> (výsledek f y x)

-- běžný take:
take 2 [10, 20, 30]
-- take s přehozenými parametry:
flipArgs take [10, 20, 30] 2

[10,20]

[10,20]

In [2]:
:t take
-- můžeme částečně aplikovat:
takeButTheOtherWayAround = flipArgs take
:t takeButTheOtherWayAround

take :: forall a. Int -> [a] -> [a]

takeButTheOtherWayAround :: forall {a}. [a] -> Int -> [a]

Ekvivalentně pomocí lambda funkce („funkce má jako parametr funkci a vrátí funkci“):

In [3]:
flipArgs' :: (a -> b -> c) -> b -> a -> c
flipArgs' f = \ x y -> f y x

Line 2: Redundant lambda
Found:
flipArgs' f = \ x y -> f y x
Why not:
flipArgs' f x y = f y xLine 2: Avoid lambda
Found:
\ x y -> f y x
Why not:
flip f

Tato funkce je dost užitečná na to, aby byla součástí standardní knihovny.

In [4]:
:t flip

flip :: forall a b c. (a -> b -> c) -> b -> a -> c

**Otázka:** Jaká je typová anotace následujících funkcí a jaké budou výsledky vyhodnocení?

In [5]:
-- usefulCalc :: ?
usefulCalc x y = x + y * 10
-- usfOne :: ?
usfOne = usefulCalc 1
-- usfTwo :: ?
usfTwo = (flip usefulCalc) 2

Line 6: Redundant bracket
Found:
(flip usefulCalc) 2
Why not:
flip usefulCalc 2

In [6]:
usefulCalc 1 0
usefulCalc 1 2
usfOne 2
usfOne 5
usfTwo 2
usfTwo 5

1

21

21

51

22

25

### Skládání funkcí
_aneb vzpomínka na IDM_

„ef po gé“: $(f\circ g)(x) = f(g(x))$ — nejprve se vyhodnotí g, potom f (proto „po“)

„ef po gé po há“: $f \circ g \circ h = (f \circ (g \circ h)) = ((f \circ g) \circ h)$
- $z = (f \circ (g \circ h))(x) \Rightarrow y = g(h(x)),\ z = f(y) \Rightarrow z = f(g(h(x))$
- $z = ((f \circ g) \circ h)(x) \Rightarrow z = f(g(y)),\ y = h(x) \Rightarrow z = f(g(h(x))$

Haskell: `(f . g) x = f (g x)`

In [7]:
-- sum sečte čísla v seznamu
sum' (x:xs) = x + sum' xs
sum' [] = 0

:t sum'
sum' [10, 20, 30]
-- negate neguje číslo
:t negate
negate 42

sum' :: forall {a}. Num a => [a] -> a

60

negate :: forall a. Num a => a -> a

-42

$negate(sum(x)) = (negate \circ sum)(x)$

In [8]:
negate . sum' $ [10, 20, 30]
(negate . sum') [10, 20, 30]

-60

-60

Operátor `.` má nižší precedenci než aplikace funkce, tudíž pokud bychom psali jen `negate . sum [...]`, uzávorkovalo by se jako `negate . (sum [...])`, což nedává smysl.

In [9]:
:t (.)
:i (.)

(.) :: forall b c a. (b -> c) -> (a -> b) -> a -> c

(.) :: (b -> c) -> (a -> b) -> a -> c 	-- Defined in ‘GHC.Base’
infixr 9 .

**Otázka:** Známe funkci `take :: Int -> [a] -> [a]` a funkci `reverse :: [a] -> [a]`. Vytvořte dvě ekvivalentní funkce `takeFive`, `takeFive'`, které ze seznamu vezmou právě 5 prvků – jednu s pomocí závorek a explicitního parametru `x`, druhou pomocí skládání funkcí:

In [10]:
takeFive :: [a] -> [a]
takeFive x = take 5 (reverse x)
takeFive' = take 5 . reverse

### HOF pro práci se seznamy

HOF se často používají pro operace typu „vem každou položku ze seznamu a proveď s ní něco“:

In [11]:
:t map
:t filter
:t zipWith

map :: forall a b. (a -> b) -> [a] -> [b]

filter :: forall a. (a -> Bool) -> [a] -> [a]

zipWith :: forall a b c. (a -> b -> c) -> [a] -> [b] -> [c]

In [12]:
map (*10) [1,2,3]
filter (>5) [1..10]
zipWith (*) [1,2,3] [100, 21, 23]

[10,20,30]

[6,7,8,9,10]

[100,42,69]

**Otázka:** Vytvořte funkci `countIf` (počítá, kolik prvků splňuje nějakou podmínku, kterou tomu strčím jako funkci) pomocí `filter` a `length`.

In [13]:
countIf :: (a -> Bool) -> [a] -> Int
countIf f = length . filter f

## Rekurze

Obligátní faktoriál:

In [14]:
factorial :: Integral a => a -> a
-- škaredější
-- factorial n = if n == 0 then 1 else n * factorial (n - 1)

-- hezčí
factorial 0 = 1  -- ZÁKLADNÍ PŘÍPAD (base case) nebo taky UKONČOVACÍ PODMÍNKA
factorial n = n * factorial (n - 1)

In [15]:
factorial 4

24

In [16]:
inc :: Num a => a -> a
inc = (1+)

inc 0

two :: Num a => a
two = inc . inc $ 0 -- = inc(inc(0)) = 1+ (1+ 0)

three :: Num a => a
three = inc . inc . inc $ 0 -- = inc(inc(inc(0)))

1

Jak to zobecnit pro parametricky zvolitelný počet opakování?

In [17]:
incTimes :: (Eq a, Num a) => Integer -> a -> a

incTimes 0     n = n
incTimes times n = let rest = incTimes (times - 1) n
                   in 1 + rest
--                    --------         

incTimes 3 0
incTimes 3 1
incTimes 2 1

-- <incTimes 2 1>
-- = <1 + [incTimes (2 - 1) 1]> = <1 + [incTimes 1 1]>
-- = <1 + [1 + (incTimes (1 - 1) 1)]> = <1 + [1 + (incTimes 0 1)]>
-- = <1 + [1 + (1)]> 
-- = 3

3

4

3

Jak to zobecnit pro použití s různými funkcemi?

In [18]:
applyTimes :: Integer -> (a -> a) -> a -> a

applyTimes 0     f start = start
applyTimes times f start = let rest = applyTimes (times - 1) f start
                           in f rest

applyTimes 3 (+1) 0


-- další možný zápis:

applyTimes 0     f start = start
applyTimes times f start = f . (applyTimes (times - 1) f) $ start

applyTimes 3 (+1) 0

Line 13: Redundant bracket
Found:
f . (applyTimes (times - 1) f)
Why not:
f . applyTimes (times - 1) f

3

3

## Seznam je položka připláclá k seznamu

Seznam je **rekurzivní datový typ** – je zadefinovaný sebereferenčním tvrzením:
- prázdný seznam je seznam,
- konstrukce (prvek připojený na začátek seznamu) je seznam,
- (nic jiného seznam není).

Přepsáno do syntaxe Haskellu:
```haskell
data List a = [] | a : (List a)
```

Pojďme si tuto definici rozparsovat:
- `data List a`: část před `=` definuje typový konstruktor. Tady vidíme kromě názvu typu `List` i nějakou typovou proměnnou: tímhle způsobem právě vyjadřujeme, že uvnitř seznamu bude _nějaký_ libovolný typ `a`.
- `=`: odtud dál už následují datové konstruktory.
- `[]`: jednou možnou realizací typu seznam je hodnota „prázdný seznam“.
- `|`: udává, že typ má více možných realizací.
- `a : [a]`: druhý datový konstruktor je definovaný infixovým způsobem jako operátor `:` a říká, že druhou možnou realizací seznamu je „prvek typu `a` připojený před existující seznam typu `a`. Operátoru `:` se říká **cons**.

Mimochodem, `:i []` by vám ukázalo `data List a = [] | a : [a]`, ale to `[a]` na konci je jen syntaktický cukr označující právě `List a`. Celkově je syntaxe seznamů ve skutečnosti jen syntaktický cukr pro řetězení těch datových konstruktorů `:`:

In [19]:
[] == []
1:[] == [1]
1:(2:[]) == [1,2]
1:(2:(3:[])) == [1,2,3]
1:2:3:[] == [1,2,3]

Line 1: Use null
Found:
[] == []
Why not:
null []Line 1: Use null
Found:
[] == []
Why not:
null []Line 2: Use list literal
Found:
1 : []
Why not:
[1]Line 3: Use list literal
Found:
1 : (2 : [])
Why not:
[1, 2]Line 4: Use list literal
Found:
1 : (2 : (3 : []))
Why not:
[1, 2, 3]Line 5: Use list literal
Found:
1 : 2 : 3 : []
Why not:
[1, 2, 3]

True

True

True

True

True

Úplně stejně se chovající datovou strukturu bychom si mohli nadefinovat i sami:

In [20]:
data MyList a = EmptyList | Cons a (MyList a) deriving Show

myListVal :: MyList Int
myListVal = Cons 1 (Cons 2 (Cons 3 EmptyList))

myListVal

Cons 1 (Cons 2 (Cons 3 EmptyList))

Teď už by měl být trochu jasnější pattern matching nad seznamy:

In [21]:
-- pro náš datový typ:
myHead :: MyList a -> a
myHead (Cons x _) = x

myTail :: MyList a -> MyList a
myTail (Cons _ xs) = xs

myHead myListVal
myTail myListVal

-- pro standardní seznamy:
myStdHead :: [a] -> a
myStdHead (x:_) = x

myStdTail :: [a] -> [a]
myStdTail (_:xs) = xs

myStdHead [1, 2, 3]
myStdTail [1, 2, 3]

1

Cons 2 (Cons 3 EmptyList)

1

[2,3]

**Otázka:** S využitím pattern matchingu definujte funkci, která řekne, zda je (standardní haskellový) seznam prázdný:

In [22]:
isEmpty :: [a] -> Bool
isEmpty [] = True
isEmpty _ = False

#### Vsuvka 1: Call me Maybe

Všimněte si, že naše funkce `myHead` a `myTail` „trpí“ stejným neduhem jako ty standardní: nejsou úplně definované – nepokrývají případ prázdného seznamu. V případě _tail_ bychom to ještě mohli vyřešit:

In [23]:
myTail :: MyList a -> MyList a
myTail (Cons _ xs) = xs
myTail EmptyList = EmptyList

U `myHead` to ale vyřešit takhle nejde: funkce má vracet hodnotu typu `a`, jenže když na vstupu dostanu `EmptyList`, nemám odkud bych tu potřebnou hodnotu vzal.

Vzpomeňte na poslední příklad v přípravě, kde jste měli definovat nový datový typ `Box a` jakožto krabici pro hodnotu s variantou „prázdná krabice“. Přesně takový typ existuje i ve standardním Haskellu, jmenuje se `Maybe` a jeho možné hodnoty jsou `Nothing` a `Just a`.

In [24]:
:i Maybe

type Maybe :: * -> *
data Maybe a = Nothing | Just a
  	-- Defined in ‘GHC.Maybe’
instance Semigroup a => Monoid (Maybe a) -- Defined in ‘GHC.Base’
instance Semigroup a => Semigroup (Maybe a) -- Defined in ‘GHC.Base’
instance Foldable Maybe -- Defined in ‘Data.Foldable’
instance Traversable Maybe -- Defined in ‘Data.Traversable’
instance Read a => Read (Maybe a) -- Defined in ‘GHC.Read’
instance Show a => Show (Maybe a) -- Defined in ‘GHC.Show’
instance Applicative Maybe -- Defined in ‘GHC.Base’
instance Functor Maybe -- Defined in ‘GHC.Base’
instance MonadFail Maybe -- Defined in ‘Control.Monad.Fail’
instance Monad Maybe -- Defined in ‘GHC.Base’
instance Eq a => Eq (Maybe a) -- Defined in ‘GHC.Maybe’
instance Ord a => Ord (Maybe a) -- Defined in ‘GHC.Maybe’

In [25]:
Nothing
Just 'a'
Nothing == Nothing
Just 12 == Just 16

Line 3: Use isNothing
Found:
Nothing == Nothing
Why not:
isNothing NothingLine 3: Use isNothing
Found:
Nothing == Nothing
Why not:
isNothing Nothing

Nothing

Just 'a'

True

False

**Otázka:** Co bude výsledkem vyhodnocení následujícího výrazu a proč?

In [26]:
Just "abc" == Just 2

: 

#### Vsuvka 2: Stráže

Pokud potřebujeme řešit víc podmínek, mohli bychom řetězit if-then-else, ale nebylo by to moc pěkné:

In [27]:
numVibe :: (Ord a, Num a) => a -> String
numVibe x = if x < 36 then "meh number" else
              if x > 45 && x < 190 then "lovely" else
                if x  > 359 then "gosh it's huge" else "whatever"

Line 2: Use guards
Found:
numVibe x
  = if x < 36 then
        "meh number"
    else
        if x > 45 && x < 190 then
            "lovely"
        else
            if x > 359 then "gosh it's huge" else "whatever"
Why not:
numVibe x
  | x < 36 = "meh number"
  | x > 45 && x < 190 = "lovely"
  | x > 359 = "gosh it's huge"
  | otherwise = "whatever"

Alternativní syntaxí jsou *stráže* (guards):

In [28]:
numVibe x
    | x < 36            = "meh number"
    | x > 45 && x < 190 = "lovely"
    | x > 339           = "gosh it's huge"
    | otherwise         = "whatever"

Lze kombinovat třeba s *where*:

In [29]:
gradeFromPoints :: (RealFrac a, Ord a) => a -> Char
gradeFromPoints x
    | y >= 0.9 = 'A'
    | y >= 0.8 = 'B'
    | y >= 0.7 = 'C'
    | y >= 0.6 = 'D'
    | y >= 0.5 = 'E'
    | y < 0.5 = 'F'
    where y = fromIntegral (round x) / 100

gradeFromPoints 49.49
gradeFromPoints 89.5

'F'

'A'

### Rekurze a pattern matching nad seznamem

Seznam je rekurzivní typ, a tak i pro práci s ním často budeme využívat rekurzi. Velmi častý vzor:
- základní případ: co vrátím pro `[]`,
- krok: co vrátím pro `(x:xs)`.

In [30]:
-- délka seznamu
lenR :: [a] -> Int
lenR [] = 0
lenR (_:xs) = 1 + lenR xs

In [31]:
lenR "ahoj"
lenR [1..10]

4

10

In [32]:
-- suma čísel v seznamu
sumR :: Num a => [a] -> a
sumR [] = 0
sumR (x:xs) = x + sumR xs

Line 3: Use foldr
Found:
sumR [] = 0
sumR (x : xs) = x + sumR xs
Why not:
sumR xs = foldr (+) 0 xs

In [33]:
sumR [1,2,3,4]

10

Proč je `(++)` zlý operátor:

In [34]:
myPlPl :: [a] -> [a] -> [a]

myPlPl [] r = r
-- myPlPl l [] = l -- může být, ale není nutné

myPlPl (l:ls) r = l : (ls `myPlPl` r)

In [35]:
myPlPl "hello " "world"

"hello world"

Operátor musí tedy během rekurze projít **celý levý seznam** prvek po prvku!

**Otázka:** Doplňte implementaci `elemR`.

In [36]:
elemR :: Eq a => a -> [a] -> Bool
elemR _ [] = False
elemR e (x:xs) = (e == x) || elemR e xs

In [37]:
elemR 'o' "ahoj"
elemR 'x' "ahoj"

True

False

**Otázka:** Doplňte implementaci `mapR`.

In [38]:
mapR :: (a -> b) -> [a] -> [b]
mapR _ [] = []
mapR f (x:xs) = f x : mapR f xs

In [39]:
mapR (+1) [1,2,3]

[2,3,4]

**Otázka:** Doplňte implementaci `filterR`.

In [40]:
filterR :: (a -> Bool) -> [a] -> [a]
filterR _ [] = []
filterR f (x:xs)
    | f x       = x : filterR f xs
    | otherwise = filterR f xs

In [41]:
filterR odd [1..8]

[1,3,5,7]

### List comprehensions, range construction


In [42]:
[1..10]  -- enumFromTo

[1,2,3,4,5,6,7,8,9,10]

In [43]:
[5,10..100]  -- enumFromThenTo

[5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100]

In [44]:
:i Enum

type Enum :: * -> Constraint
class Enum a where
  succ :: a -> a
  pred :: a -> a
  toEnum :: Int -> a
  fromEnum :: a -> Int
  enumFrom :: a -> [a]
  enumFromThen :: a -> a -> [a]
  enumFromTo :: a -> a -> [a]
  enumFromThenTo :: a -> a -> a -> [a]
  {-# MINIMAL toEnum, fromEnum #-}
  	-- Defined in ‘GHC.Enum’
instance Enum Double -- Defined in ‘GHC.Float’
instance Enum Float -- Defined in ‘GHC.Float’
instance Enum () -- Defined in ‘GHC.Enum’
instance Enum Bool -- Defined in ‘GHC.Enum’
instance Enum Char -- Defined in ‘GHC.Enum’
instance Enum Int -- Defined in ‘GHC.Enum’
instance Enum Integer -- Defined in ‘GHC.Enum’
instance Enum Ordering -- Defined in ‘GHC.Enum’
instance Enum a => Enum (Solo a) -- Defined in ‘GHC.Enum’
instance Enum Word -- Defined in ‘GHC.Enum’

In [45]:
[ x^2 | x <- [1..10] ]

[1,4,9,16,25,36,49,64,81,100]

In [46]:
[ x^2 | x <- [1..10], even x ]

[4,16,36,64,100]

In [47]:
[ x^y | x <- [1..3], y <- [1..3] ]

[1,1,1,2,4,8,3,9,27]

In [48]:
[ (x,y,z) | x <- [1..5], y <- [6,7,0], z <- [4,5] ]

[(1,6,4),(1,6,5),(1,7,4),(1,7,5),(1,0,4),(1,0,5),(2,6,4),(2,6,5),(2,7,4),(2,7,5),(2,0,4),(2,0,5),(3,6,4),(3,6,5),(3,7,4),(3,7,5),(3,0,4),(3,0,5),(4,6,4),(4,6,5),(4,7,4),(4,7,5),(4,0,4),(4,0,5),(5,6,4),(5,6,5),(5,7,4),(5,7,5),(5,0,4),(5,0,5)]

### Do nekonečna a ještě dál

In [49]:
take 5 [1..]

[1,2,3,4,5]

In [50]:
take 9 $ drop 5 [y^2 | y <- [12..]]

[289,324,361,400,441,484,529,576,625]

Seznam `[1, 2, 3]` je tedy `1:(2:(3:[])))`. Ve skutečnosti jde vždycky o jakousi _buňku_, která obsahuje dvě položky: hodnotu a zbytek seznamu. Můžeme tedy tuto reprezentaci zapsat jako binární strom:
```
 : ◁───────┐
/ \        │
1  : ◁─────┤
  / \      │
  2  : ◁───┤
    / \    spine
    3 []
```
Hodnota typu seznam má tedy jakousi _strukturu_, ve které teprve bydlí hodnoty (resp. podvýrazy).

(Níže je jen zhuštěná forma [tohoto článku](https://en.wikibooks.org/wiki/Haskell/Laziness#Thunks_and_Weak_head_normal_form) – doporučuji pečlivě pročíst při samostudiu.)

V Haskellu se nic nevyhodnocuje, dokud nemusí. Co to přesně znamená?

> At first glance, we might think that lazy evaluation makes programs more efficient. After all, what can be more efficient than not doing anything? In practice, however, laziness often introduces an overhead that leads programmers to hunt for places where they can make their code more strict. The real benefit of laziness is in making the right things efficient enough. Lazy evaluation allows us to write more simple, elegant code than we could in a strict environment.

Líné vyhodnocování (lazy evaluation) = technika („jak“) pro dosažení [nestriktní sémantiky (non-strict semantics)](https://wiki.haskell.org/Non-strict_semantics) výpočtu („co“). Vyhodnocuje se co nejmíň a vyhodnocování se odkládá co nejdál.

```haskell
-- a)
let (x, y) = (length [1..5], reverse "olleh") in [nějaký výraz využívající x a y]
-- b)
let z = (length [1..5], reverse "olleh") in [nějaký výraz využívající x a y]
```

- Nevyhodnocujeme `length` ani `reverse`, dokud to nepotřebujeme.
- Na levé straně a) je pattern matching, který se dívá na datový konstruktor `(,)` $\Rightarrow$ `x` a `y` jsou **thunk**s – nevyhodnocené hodnoty s „receptem“ k vyhodnocení (výpočetním grafem).
- Na levé straně b) je jen proměnná $\Rightarrow$ nemusím se dívat ani dovnitř $\Rightarrow$ celé `z` je thunk.

```haskell
let z     = (length [1..5], reverse "olleh")
   (n, s) = z 
   'h':ss = s
in [nějaký výraz využívající x a y]
```

- Výrazy jsou jako cibule, mají vrstvy.
- Na začátku je `z` prostě jen thunk,
- Pro pattern match `(n, s) = z` je nutné vyhodnotit, že v `z` je dvojice (datový konstruktor `(,)`) $\Rightarrow$ vyhodnotí se **struktura** `z` a zjistí se, že jde o `(*thunk*, *thunk*)` $\Rightarrow$ `n` a `s` jsou thunks.
- Pro pattern match `'h':ss = s` potřebuje vyhodnotit, že v `s` je datový konstruktor tvorby seznamu `:` (cons) $\Rightarrow$ vyhodnotí se struktura `s` a zjistí se, že jde o `*thunk* : *thunk*`
- $\Rightarrow$ vyhodnotí se první thunk, protože se matchuje na konkrétní hodnotu.
- Zbytek zůstal nevyhodnocený, takže máme thunks `ss` a `n`.

„Vyhodnocování struktury“ $\sim$ vyhodnocování do _Weak Head Normal Form_ (WHNF, [obrázek](https://upload.wikimedia.org/wikipedia/commons/f/fc/Thunk-layers.png)). Výraz je ve WHNF, pokud jde o datový konstruktor nebo lambda abstrakci (ne aplikace).
- `(1 + 1, 2 + 2)` WHNF,
- `\x -> 2 * 2` WHNF,
- `'h' : ("ello " ++ "world")` WHNF,
- `show 123` není ve WHNF!

V _Normal Form_ je výraz, pokud už v něm nikde opravdu nejde nic vyhodnotit. 

> In fact, the only place that Haskell values get evaluated is in pattern matches and inside certain primitive IO functions.

- Funkce mohou být v *jednotlivých* argumentech _lazy_ nebo _strict_ (= vyhodnotí argument alespoň do WHNF).
- Funkce `f` je „striktnější“ než `g`, pokud `f x` vyhodnotí `x` do větší hloubky než `g x`.
- `length` např. potřebuje vyhodnotit jen strukturu: z `*thunk* : (*thunk* : (*thunk* : []))` jednoznačně poznáme, že délka seznamu je 3. 
- `f x y = length $ show x` je v `x` striktní a v `y` lazy.

> The word thunk was invented by an informal working group that was discussing the implementation of call-by-name in Algol 60. They observed that most of the analysis of (thinking about) the expression could be done at compile time; thus, at run time, the expression would already have been *thunk about* (Ingerman et al. 1960).

In [51]:
length [1, 2, undefined, 4]
let (x, y) = (4, undefined) in x
head (4 : undefined)

take 2 $ map (+2) [2, 2, undefined]

4

4

4

[4,4]

In [52]:
-- Oproti tomu: 
length [1, 2] ++ undefined + [3]

: 

**Otázka:** Jsou následující výrazy ve WHNF, NF, nebo ani v jednom?
1. `[1, 2, 3, 4, 5]`
2. `1 : 2 : 3 : 4 : (5 + 6)`
3. `enumFromTo 1 10`
4. `length [1, 2, 3, 4, 5]`
5. `sum (enumFromTo 1 10)`
6. `['a'..'m'] ++ ['n'..'z']`
7. `("abcde" !! 2, 'b')`

## Vlastní rekurzivní datové typy

Vlastní datový typ níže reprezentuje binární strom, kde každý uzel nese hodnotu a každý nelistový uzel má právě dva potomky.

In [53]:
data Tree a
  = Leaf a
  | Branch (Tree a) a (Tree a)
  deriving (Eq, Show)

**Otázka:** Jak by se dal reprezentovat strom, který nemusí mít vždycky právě dva potomky?

In [54]:
t1 :: Tree Int
t1 =
  Branch
    (Branch (Leaf 60) 2 (Branch (Leaf 50) 4 (Leaf 40)))
    5
    (Branch (Branch (Leaf 10) 7 (Leaf 20)) 9 (Leaf 30))

t1

Branch (Branch (Leaf 60) 2 (Branch (Leaf 50) 4 (Leaf 40))) 5 (Branch (Branch (Leaf 10) 7 (Leaf 20)) 9 (Leaf 30))

![strom](Lab02-tree.png)

Velikost stromu (počet uzlů):

In [55]:
sizeT :: Tree a -> Int
sizeT (Leaf _) = 1
sizeT (Branch l _ r) = 1 + sizeT l + sizeT r

In [56]:
sizeT t1

11

**Otázka:** Jak doplnit funkci pro získání výšky stromu?

In [57]:
heightT :: Tree a -> Int
heightT (Leaf _) = 0
heightT (Branch l _ r) = 1 + max (heightT l) (heightT r)

In [58]:
heightT t1

3

In-order průchod stromu (vytvoří seznam hodnot v in-order pořadí).

In [59]:
inorderT :: Tree a -> [a]
inorderT (Leaf v) = [v]
inorderT (Branch l v r) = inorderT l ++ (v : inorderT r)

In [60]:
inorderT t1

[60,2,50,4,40,5,10,7,20,9,30]

**Cvičení** na doma: udělejte si i pre-order a post-order průchody.

Mohli bychom pro náš strom definovat ekvivalent toho, co dělá `map` pro seznamy – vytvoří nový strom se stejnou _strukturou_, ve kterém na všechny _hodnoty_ aplikuje dodanou funkci.

In [61]:
preorderT :: Tree a -> [a]
preorderT (Leaf v) = [v]
preorderT (Branch l v r) = v : (preorderT l ++ preorderT r)

postorderT :: Tree a -> [a]
postorderT (Leaf v) = [v]
postorderT (Branch l v r) = postorderT l ++ postorderT r ++ [v]

preorderT t1
postorderT t1

[5,2,60,4,50,40,9,7,10,20,30]

[60,50,40,4,2,10,20,7,30,9,5]

In [62]:
:t map

map :: forall a b. (a -> b) -> [a] -> [b]

In [63]:
-- povšimněte si podobnosti s map
mapT :: (a -> b) -> Tree a -> Tree b

mapT f (Leaf v) = Leaf $ f v
mapT f (Branch l v r) = Branch (mapT f l) (f v) (mapT f r)

In [64]:
mapT (+1) t1
mapT (\x -> fromIntegral x * 12.5) t1

Branch (Branch (Leaf 61) 3 (Branch (Leaf 51) 5 (Leaf 41))) 6 (Branch (Branch (Leaf 11) 8 (Leaf 21)) 10 (Leaf 31))

Branch (Branch (Leaf 750.0) 25.0 (Branch (Leaf 625.0) 50.0 (Leaf 500.0))) 62.5 (Branch (Branch (Leaf 125.0) 87.5 (Leaf 250.0)) 112.5 (Leaf 375.0))

V Haskellu velmi rádi zobecňujeme. Proč mít jednu funkci `map` pro seznamy a druhou funkci `mapT` pro strom, když obě v podstatě dělají totéž, jen nad jinou obalovací strukturou? (Stejně jako nemáme zvlášť „iplus“ pro Int a „fplus“ pro Float...)

Řešením jsou, jak už možná tušíme, typové třídy. Typová třída `Functor f` značí datové struktury, pro které dává smysl existence nějakého ekvivalentu `map` – jde o takový jednoduchý interface pro „boxíky s hodnotou či více hodnotami“. Datové typy, které chtějí být instancí `Functor`, musí podporovat funkci `fmap`:

In [65]:
:t fmap

fmap :: forall (f :: * -> *) a b. Functor f => (a -> b) -> f a -> f b

Instancí `Functor` je i standardní seznam, takže pro seznamy `map` = `fmap`.

In [66]:
map (*2) [1..10] == fmap (*2) [1..10]

True

Udělejme i z našeho `Tree` instanci `Functor`:

In [67]:
instance Functor Tree where
    fmap = mapT

Teď už můžeme konzistentním způsobem mapovat jak seznamy, tak stromy.

In [68]:
fmap (+1) [1, 2, 3]
fmap (+1) (Branch (Leaf 1) 2 (Leaf 3))

[2,3,4]

Branch (Leaf 2) 3 (Leaf 4)

Instancí `Functor` je mimochodem třeba i `Maybe`. Zamyslete se nad tím – opět jde o nějaký typ, který v sobě uvnitř uchovává nějaké hodnoty, tak proč by nemohl podporovat operaci „prožeň všechny své vnitřní hodnoty touto funkcí“.

In [69]:
fmap (*1000) (Just 5)
fmap (*1000) Nothing

Just 5000

Nothing

Vzpomeňme na operátor `$`. Ačkoliv jsme si říkali, že „nic nedělá“, mohli bychom jeho implementaci `f $ b = f b` přečíst jako: „Vezmi funkci `f` a *aplikuj* ji na hodnotu `b`.“

In [70]:
:t ($)
-- f $ b = f b
(+1) $ 10  -- vezme funkci (+1) a aplikuje ji na 10

($) :: forall a b. (a -> b) -> a -> b

11

Existuje i podobný operátor `<$>`, který je definovaný pro `Functor` typy a dá se podobným způsobem číst jako: „Vezmi funkci `f` a *aplikuj* ji **dovnitř** boxíku – na hodnotu/hodnoty, které jsou uvnitř toho boxíku.“ Jde tedy jen o syntaktickou zkratku pro `fmap f b`:

In [71]:
:t (<$>)
-- f <$> b = fmap f b

(+1) <$> [10, 20, 30]
(+1) <$> (Branch (Leaf 1) 2 (Leaf 3))

(<$>) :: forall (f :: * -> *) a b. Functor f => (a -> b) -> f a -> f b

[11,21,31]

Branch (Leaf 2) 3 (Leaf 4)

### Binární vyhledávací strom

Povšimněte si, že tato reprezentace binárního stromu se od předchozí liší – zde máme datové konstruktory `Node`, který reprezentuje uzel s klíčem a hodnotou, a `Empty`, který vlastně reprezentuje „neuzel“ (absenci potomka). To je opět jen nějaké vaše reprezentační rozhodnutí – stromy mohou mít mnoho podob podle toho, jak s nimi pak chcete pracovat.

Vzpomeňte, že v BST platí: vlevo menší klíče, vpravo větší.

In [72]:
data BST k v
  = Empty
  | Node (BST k v) k v (BST k v)
  deriving Eq  -- záměrně nederivujeme Show – níže si ho uděláme ručně

**Otázka:** Doplňte funkci pro hledání hodnoty v BST.

In [73]:
lookupBST :: Ord k => k -> BST k v -> Maybe v

lookupBST _ Empty = Nothing
lookupBST k (Node l ck cv r)
    | k < ck    = lookupBST k l
    | k > ck    = lookupBST k r
    | otherwise = Just cv

V Haskellu žádné hodnoty nejsou „mutable“ – funkce typu „insert“ nešolichají s nějakými ukazateli. Pokud chceme reprezentovat „vložení do stromu“, bude to funkce, která rekurzivně projde stávající strom a bude přitom vracet jeho upravenou variantu.

**Cvičení na doma:** Doplňte funkci `insertBST`.

In [74]:
-- „vložení do BST“: 
--              vezmi  klíč    hodnotu    existující strom
--                     ↓       ↓          ↓            a vrať modifikovaný strom, ve kterém bude ta hodnota pod daným klíčem uložená
insertBST :: Ord k =>  k    -> v       -> BST k v          -> BST k v

-- „vkládám“ na místo, kde nic nebylo -> nahrazuju toto místo novou instancí uzlu
insertBST k v Empty = Node Empty k v Empty
insertBST k v (Node l ck cv r)  -- pokud vkládám na místo, kde je uzel s klíčem cK...
  | k < ck    = Node (insertBST k v l) ck cv r  -- ... a vkládaný klíč k je menší cK, pak volám funkci rekurzivně pro levý podstrom;
  | k > ck    = Node l ck cv (insertBST k v r)  -- ... a vkládaný klíč k je větší než cK, pak volám funkci rekurzivně pro pravý podstrom;
  | otherwise = Node l ck v r  -- ... a vkládaný klíč k je roven cK, pak vložím hodnotu do daného uzlu.

In [75]:
-- vytvoření konkrétní instance BST
bst1 :: BST Int Char
bst1 =
  insertBST 5 'a' $
  insertBST 2 'b' $
  insertBST 9 'c' $
  insertBST 4 'x' $
  insertBST 7 'a' Empty

In [76]:
-- in-order průchod vypadá dost podobně
-- zde jsem si zvolil, že chci, aby ta funkce vracela seznam dvojic klíč-hodnota, ale mohl bych to udělat třeba jen pro hodnoty

inord :: BST k v -> [(k,v)]
inord Empty = []
inord (Node l k v r) = inord l ++ ((k, v) : inord r)

Vyzkoušíme naše funkce:

In [77]:
inord bst1

[(2,'b'),(4,'x'),(5,'a'),(7,'a'),(9,'c')]

In [78]:
lookupBST 7 bst1
lookupBST 100 bst1

Just 'a'

Nothing

Pokud bychom ale chtěli strom zobrazit, teď to nepůjde, protože jsme nepoužili `deriving Show`:

In [79]:
bst1

: 

Pojďme si nyní ručně nadefinovat (malinko) hezčí způsob, jak se ze stromů dají dělat textové reprezentace:

In [80]:
-- "Showable" mohou být z principu jen stromy, kde klíč i hodnota jsou taky Show
-- proto v definici instance typové třídy definuju omezení na typy k, v uvnitř stromu
instance (Show k, Show v) => Show (BST k v) where
    -- budu si chtít držet úroveň zanoření
    show l = go l 0 where
            -- prázdný neuzel nemá žádnou reprezentaci
            go Empty _ = ""
            -- uzel chci tisknout tak, že budu odsazovat podle úrovně zanoření
            go (Node l k v r) d = 
                -- a tady jen skládám dohromady delší řetězec pomocí ++
                let d' = d + 1 in 
                     (if d == 0 then "" else "\n")  -- newline (pokud nejsem v kořeni)
                  ++ replicate (d*2) ' '            -- odsazení
                  ++ show k                         -- klíč
                  ++ ":"                            -- oddělovač
                  ++ show v                         -- hodnota
                  ++ go l d'                        -- levý podstrom 
                  ++ go r d'                        -- pravý podstrom

In [81]:
Node (Node Empty 1 'b' Empty) 2 'a' (Node (Node Empty 4 'd' Empty) 5 'c' Empty)

2:'a'
  1:'b'
  5:'c'
    4:'d'

Vzpomeňte ale, že `(++)` je zlý operátor – jeho použití vyžaduje rekurzivní průchod celým levým seznamem (jeho časová složitost je $\mathcal{O}(n)$ vůči délce levého argumentu). Nemilé je, že ho tady vyhodnotíme celkem šestkrát, takže první seznam se celý rekurzivně projde šestkrát, druhý pětkrát a tak dále. Demonstrativně:

In [82]:
(((['a', 'b'] ++ ['c', 'd']) ++ ['e']) ++ ['f'])
-- 1. projedu a, b, abych je připojil k ['c', 'd']
-- 2. projedu a, b, c, d, abych je připojil k ['e']
-- 3. projedu a, b, c, d, e, abych je připojil k ['f']
-- => hodnot a, b jsem se dotkl TŘIKRÁT

"abcdef"

Typová třída `Show` na tento problém myslí a její instance mohou (měly by) definovat radši `showsPrec`:

In [83]:
:t showsPrec
:i ShowS

showsPrec :: forall a. Show a => Int -> a -> ShowS

type ShowS :: *
type ShowS = String -> String
  	-- Defined in ‘GHC.Show’

Číselný parametr `showsPrec` zatím vůbec neřešme a pro jednoduchost řekněme, že to je jen funkce `shows :: Show a => a -> ShowS`. Co teda dělá `shows` a co má být `ShowS`?

`shows` vezme věc X a vrátí funkci, která po aplikaci na řetězec Y vrátí řetězec X konkatenovaný s Y.\
`ShowS = String -> String` je „string builder“: je to **funkce**, která od někoho vezme „zbytek výstupu“ a před něj přidá svůj kus textu.

In [84]:
showsFoo :: ShowS  -- alias pro (String -> String)
-- je to funkce, která při svém vyhodnocení přikonkatenuje 
-- nějaký String k něčemu, co do ní strčím
showsFoo = (\y -> "Foo" ++ y)

showsFoo ""  -- vypíše jen "Foo"
showsFoo "."  -- vypíše "Foo."

Line 4: Redundant lambda
Found:
showsFoo = (\ y -> "Foo" ++ y)
Why not:
showsFoo y = "Foo" ++ yLine 4: Avoid lambda
Found:
(\ y -> "Foo" ++ y)
Why not:
("Foo" ++)Line 4: Redundant bracket
Found:
(\ y -> "Foo" ++ y)
Why not:
\ y -> "Foo" ++ y

"Foo"

"Foo."

Díky tomuto zdánlivému nesmyslu však můžeme efektivně skládat řetězce pomocí našeho oblíbeného **skládání funkcí** `(.)`:

In [85]:
:t shows 1234
:t showsFoo

(showsFoo . shows 1234) ""

shows 1234 :: ShowS

showsFoo :: ShowS

"Foo1234"

1. `shows 1234` vyrobí funkci „`připoj_1234_před(x)`“
2. `showsFoo` vyrobí funkci „`připoj_Foo_před(x)`
3. `showsFoo . shows 1234` vyrobí složenou funkci `připoj_Foo_před(připoj_1234_před(x))`
4. Celou tuhle funkci použiju na argument `""`.

Jak se to teď bude vyhodnocovat? Funkce se budou vyhodnocovat „zevnitř“: nejprve se teda vyhodnotí funkce `připoj_1234_před("")`, která (nějak transformuje to číslo na řetězec `"1234"` a) *jednou* projde seznam `['1', '2', '3', '4']`, aby ho připojila k `""` $\Rightarrow$ výsledkem bude řetězec `"1234"`. Pak se vyhodnotí funkce `showsFoo`, která *jednou* projde seznam `['F', 'o', 'o']`, aby ho přikonkatenovala k výsledku té vnořené funkce, tedy `"1234"`. Vidíme, že každý seznam v „řetězci volání“ se projde právě jednou.

Jak by tedy mohla vypadat `showsPrec` pro náš strom?

In [86]:
instance (Show k, Show v) => Show (BST k v) where
    showsPrec n Empty = id  -- nic nepřidá: \s -> s

    -- Čtěte zleva doprava: každá část je ShowS (String -> String),
    -- kompozice (.) je skládá do jednoho „builderu“.
    showsPrec n (Node l k v r) =
          (if n == 0 then id else ('\n':))  -- newline mimo kořen
        . (replicate (n*2) ' ' ++)          -- indentace (prefix)
        . shows k                           -- klíč
        . (':':)                            -- oddělovač
        . shows v                           -- hodnota
        . showsPrec n' l                    -- levý podstrom
        . showsPrec n' r                    -- pravý podstrom
      where
        n' = n + 1

In [87]:
bst1

7:'a'
  4:'x'
    2:'b'
    5:'a'
  9:'c'

Standardní význam parametru `n` v `showsPrec n` je *precedence kontextu*, používá se pro řešení závorek např. při tisknutí výrazů (pomocí toho parametru si pak pamatujete precedenci, podle které selektivně přidáváte do builderu závorky). My jsme ho tady ale zneužili jako ten počítač hloubky zanoření ve stromu (pro účely odsazení). Nikdo nám v tom nemůže zabránit!